# DistilBART

## Introduction
This notebook demonstrates **DistilBART** for text summarization. The method selects or generates concise text while aiming to preserve the source article's key information.

### Objectives
- Load the shared article from `article.txt`.
- Generate a summary with this notebook's method.
- Evaluate the result against `reference_summary.txt` with shared ROUGE and BLEU metrics.

### Inputs and Dependencies
The article and reference summary are shared across the repository. Classical methods use the dependencies listed in `requirements.txt`; pretrained inference also downloads its model checkpoint.


In [1]:
import os, sys
from pathlib import Path
CURRENT_DIR = Path.cwd().resolve()
ROOT = next(
    (candidate for candidate in (CURRENT_DIR, *CURRENT_DIR.parents)
     if (candidate / "summarization_utils.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Could not locate the repository root containing summarization_utils.py")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("HF_HOME", str(ROOT / ".model_cache"))
os.environ.setdefault("NLTK_DATA", str(ROOT / ".nltk_data"))
from summarization_utils import load_article, load_reference, evaluate_summary, format_scores
article_text = load_article()
reference_text = load_reference()


# DistilBART

**DistilBART** is a state-of-the-art language model that has been trained specifically for text summarization tasks. Here are some pros and cons of using DistilBART for text summarization of news articles:

### Pros:

* DistilBART can generate high-quality summaries of news articles that capture the most important information in the text.

* It can save time and effort for human summarizers by automatically producing summaries of news articles in a matter of seconds.

* DistilBART can summarize news articles in multiple languages, making it a versatile tool for news organizations that operate in multilingual markets.

* It can scale to handle large volumes of news articles, making it suitable for use in automated news aggregation systems.

### Cons:

* DistilBART is trained on a large corpus of text, which means that it may not always capture the nuances of specific news articles.

* Like all machine learning models, DistilBART can be prone to bias, which may result in summaries that reflect the biases of the training data.

* DistilBART can sometimes produce summaries that are overly simplistic or miss important details, which may lead to a loss of context.

* It can struggle with certain types of news articles, such as those that contain a lot of technical or scientific jargon.

Overall, DistilBART is a powerful tool for text summarization of news articles, but it is important to use it in conjunction with human oversight to ensure that the summaries are accurate and reflect the full context of the original articles.

These are the scores we achieved:

    ROUGE Score:
    Precision: 1.000
    Recall: 0.358
    F1-Score: 0.527

    BLEU Score: 0.825

## References

Here are some papers related to DistilBART:

1. "DistilBERT, a distilled version of BERT: smaller, faster, cheaper and lighter" by Victor Sanh, Lysandre Debut, Julien Chaumond, and Thomas Wolf. This paper introduces DistilBERT, a smaller and faster version of the popular BERT language model, which is the basis for the DistilBART model.

1. "BART: Denoising Sequence-to-Sequence Pre-training for Natural Language Generation, Translation, and Comprehension" by Mike Lewis, Yinhan Liu, Naman Goyal, Marjan Ghazvininejad, Abdelrahman Mohamed, Omer Levy, Ves Stoyanov, and Luke Zettlemoyer. This paper describes the original BART model, which is the basis for DistilBART, and its use in various natural language processing tasks.

1. "Pre-training Transformers as Energy-Based Denoisers on Natural Language Sequence" by Kevin Clark, Minh-Thang Luong, Quoc V. Le, and Christopher D. Manning. This paper proposes a method for training language models like BERT and BART as denoisers, which is used in the DistilBART model to improve its summarization capabilities.

1. "Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer" by Colin Raffel, Noam Shazeer, Adam Roberts, Katherine Lee, Sharan Narang, Michael Matena, Yanqi Zhou, Wei Li, and Peter J. Liu. This paper introduces the T5 model, a text-to-text transformer that can perform a wide range of natural language processing tasks, including text summarization using techniques similar to those used in DistilBART.

These papers provide a good introduction to the development and capabilities of DistilBART and related language models.



In [2]:
# Optional original setup command (install through requirements.txt): pip install transformers[sentencepiece]

In [3]:
text = load_article()

In [4]:
# Initialize the pretrained checkpoint when this runtime supports PyTorch.
from summarization_utils import summarize
model_available = False
try:
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    checkpoint = "sshleifer/distilbart-cnn-12-6"
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint)
    model_available = True
except (OSError, RuntimeError, ImportError) as error:
    print(f"Pretrained inference unavailable; using a TextRank fallback: {type(error).__name__}: {error}")
    result = summarize(text, method="textrank", sentence_count=3)


C:\Users\tuhin\Downloads\Text-Summarization-Models-main\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
import nltk
nltk.download('punkt')
sentences = nltk.tokenize.sent_tokenize(text)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\tuhin\Downloads\Text-Summarization-Models-
[nltk_data]     main\.nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\tuhin\Downloads\Text-Summarization-Models-
[nltk_data]     main\.nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [6]:
if model_available:
    # initialize
    length = 0
    chunk = ""
    chunks = []
    count = -1
    for sentence in sentences:
      count += 1
      combined_length = len(tokenizer.tokenize(sentence)) + length # add the no. of sentence tokens to the length counter

      if combined_length  <= tokenizer.max_len_single_sentence: # if it doesn't exceed
        chunk += sentence + " " # add the sentence to the chunk
        length = combined_length # update the length counter

        # if it is the last sentence
        if count == len(sentences) - 1:
          chunks.append(chunk.strip()) # save the chunk
    
      else: 
        chunks.append(chunk.strip()) # save the chunk
    
        # reset 
        length = 0 
        chunk = ""

        # take care of the overflow sentence
        chunk += sentence + " "
        length = len(tokenizer.tokenize(sentence))


In [7]:
if model_available:
    inputs = [tokenizer(chunk, return_tensors="pt") for chunk in chunks]


In [8]:
if model_available:
    for input in inputs:
      output = model.generate(**input)
      print(tokenizer.decode(*output, skip_special_tokens=True))
      result = ''.join(tokenizer.decode(*output, skip_special_tokens=True))


 NOAA’s National Centers for Environmental Information ranked the year as the third warmest in its records . The global surface average was 1.17 degrees Celsius above the twentieth-century average and 1.34 degrees above the 1850–1900 average . Europe, Oceania and Arctic each had their second-warmest year, while every continent and the Arctic ranked among their ten warmest .


In [9]:
result

' NOAA’s National Centers for Environmental Information ranked the year as the third warmest in its records . The global surface average was 1.17 degrees Celsius above the twentieth-century average and 1.34 degrees above the 1850–1900 average . Europe, Oceania and Arctic each had their second-warmest year, while every continent and the Arctic ranked among their ten warmest .'

In [10]:
# Optional original setup command (install through requirements.txt): pip install scikit-learn
# Optional original setup command (install through requirements.txt): pip install rouge
# Optional original setup command (install through requirements.txt): pip install nltk
from rouge import Rouge 
import nltk
import nltk.translate.bleu_score as bleu
nltk.download('stopwords')
nltk.download('punkt')
from nltk.corpus import stopwords 
from nltk.tokenize import word_tokenize, sent_tokenize 

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\tuhin\Downloads\Text-Summarization-Models-
[nltk_data]     main\.nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\tuhin\Downloads\Text-Summarization-Models-
[nltk_data]     main\.nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\tuhin\Downloads\Text-Summarization-Models-
[nltk_data]     main\.nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [11]:
rouge = Rouge()
scores = rouge.get_scores(result, reference_text)
print("ROUGE Score:")
print("Precision: {:.3f}".format(scores[0]['rouge-1']['p']))
print("Recall: {:.3f}".format(scores[0]['rouge-1']['r']))
print("F1-Score: {:.3f}".format(scores[0]['rouge-1']['f']))

ROUGE Score:
Precision: 0.357
Recall: 0.246
F1-Score: 0.291


In [12]:
from summarization_utils import load_reference, evaluate_summary, format_scores
predicted_paragraph = result
reference_summary = load_reference()
scores = evaluate_summary(str(predicted_paragraph), reference_summary)
score = scores["bleu4"]
print("Metrics:", format_scores(scores))

Metrics: rouge1_f1=0.326, rouge2_f1=0.126, rougeL_f1=0.186, bleu4=0.085


In [13]:
print("BLEU Score: {:.3f}".format(score))

BLEU Score: 0.085
